# REE 2019–2025 → InfluxDB → Grafana

Este notebook convierte las **7 preguntas principales del proyecto** en datos analizables y, además, prepara esos resultados para poder visualizarlos en **InfluxDB y Grafana**.

## Flujo del proyecto

**CSV → Pandas/Jupyter → tablas y gráficos → InfluxDB → Grafana**

Las preguntas principales son:

1. P1 — Renovable frente a nuclear dentro de la generación sin emisiones.
2. P2 — Renovable frente a generación con emisiones.
3. P3 — Tecnologías con crecimiento estructural frente a evolución irregular.
4. P4 — Potencia instalada frente a generación.
5. P5 — Generación frente a demanda por comunidad autónoma.
6. P6 — Hidráulica frente a generación con emisiones.
7. P7 — Intercambios eléctricos con Francia, Portugal, Marruecos y Andorra.

**P0 — Calidad de datos** se utiliza como control previo.

---

## Qué hace este notebook

- Calcula los indicadores de las preguntas.
- Genera tablas y gráficos.
- Guarda cada resultado en CSV.
- Crea una tabla maestra preparada para InfluxDB.
- Puede escribir esa tabla directamente en InfluxDB 2.x.
- Incluye consultas **Flux** listas para copiar en Grafana.

> Primero ejecuta todo hasta la sección de InfluxDB y comprueba las tablas. Después configura tus datos de InfluxDB y ejecuta la escritura.

## 0. Instalación de librerías

Si ya tienes las librerías instaladas, puedes saltarte esta celda.

Para InfluxDB utilizamos el cliente oficial de Python para InfluxDB 2.x.

In [ ]:
# Ejecuta esta celda si necesitas instalar las librerías

%pip install pandas numpy matplotlib influxdb-client

In [ ]:
import os
import re
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

ARCHIVO = "dataset_ree_limpio_transicion_energetica_2019_2025_csv.csv"

df = pd.read_csv(ARCHIVO)
df["date"] = pd.to_datetime(df["date"], errors="coerce")

print("Registros:", len(df))
print("Fecha mínima:", df["date"].min())
print("Fecha máxima:", df["date"].max())

# 1. P0 — Control de calidad

Antes de enviar datos a InfluxDB comprobamos los problemas principales:

- registros marcados para revisión,
- duplicados,
- unidades,
- granularidad,
- territorios.

In [ ]:
print("=== QUALITY FLAG ===")
print(df["quality_flag"].value_counts(dropna=False))

print("\n=== DUPLICADOS ===")
print(df.duplicated().sum())

print("\n=== UNIDADES ===")
print(df["value_unit"].value_counts(dropna=False))

print("\n=== GRANULARIDAD ===")
print(df["time_granularity"].value_counts(dropna=False))

print("\n=== GEOGRAFÍAS ===")
print(df["geography_type"].value_counts(dropna=False))

In [ ]:
# Registros marcados para revisar

revision = df[df["quality_flag"] != "ok"].copy()

print("Registros que requieren revisión:", len(revision))
display(revision.head(20))

# 2. Preparación de las salidas

Todos los resultados que queramos enviar a InfluxDB tendrán esta estructura:

- `fecha`: momento de la observación.
- `pregunta`: P1, P2, ..., P7.
- `metrica`: nombre de la métrica.
- `tecnologia`: cuando corresponda.
- `territorio`: cuando corresponda.
- `pais`: cuando corresponda.
- `valor`: valor numérico.

En InfluxDB utilizaremos:

**measurement:** `ree_analisis`

**tags:** `pregunta`, `metrica`, `tecnologia`, `territorio`, `pais`

**field:** `valor`

Esto permite filtrar fácilmente desde Grafana.

In [ ]:
SALIDA = "salidas_influx_ree"
os.makedirs(SALIDA, exist_ok=True)

salidas = []

def guardar_salida(tabla, pregunta, nombre):
    tabla = tabla.copy()
    tabla["pregunta"] = pregunta
    ruta = os.path.join(SALIDA, nombre + ".csv")
    tabla.to_csv(ruta, index=False, encoding="utf-8-sig")
    salidas.append(tabla)
    print("Guardado:", ruta, "| filas:", len(tabla))
    return tabla

# 3. P1 — Renovable vs. nuclear

### Pregunta

**¿Cómo ha cambiado la composición de la generación sin emisiones entre 2019 y 2025 y qué parte de ese cambio procede de las energías renovables frente a la nuclear?**

Usamos directamente el endpoint `evolucion_generacion_emisiones`, que contiene:

- `Sin emisiones de CO2 eq.`
- `Con emisiones de CO2 eq.`

Y el endpoint `evolucion_renovable_no_renovable`, que contiene:

- `Renovable`
- `No renovable`

Para comparar renovable y nuclear, utilizamos la estructura de generación por tecnología.

In [ ]:
# P1: generación renovable y nuclear por año.
# Trabajamos con el sistema eléctrico nacional.

p1_base = df[
    (df["geography_type"] == "system") &
    (df["endpoint"] == "estructura_generacion") &
    (df["value_unit"] == "MWh") &
    (df["indicator_name"].isin(["Eólica", "Solar fotovoltaica", "Hidráulica",
                                "Solar térmica", "Otras renovables",
                                "Residuos renovables", "Nuclear"]))
].copy()

p1_anual = (
    p1_base.groupby(["year", "indicator_name"], as_index=False)["value"]
    .sum()
)

p1_anual.head(20)

In [ ]:
# Agrupamos las tecnologías renovables.

renovables_p1 = [
    "Eólica", "Solar fotovoltaica", "Hidráulica",
    "Solar térmica", "Otras renovables", "Residuos renovables"
]

p1_anual["grupo"] = np.where(
    p1_anual["indicator_name"].isin(renovables_p1),
    "Renovable",
    "Nuclear"
)

p1_grupos = (
    p1_anual.groupby(["year", "grupo"], as_index=False)["value"]
    .sum()
)

p1_tabla = (
    p1_grupos.pivot(index="year", columns="grupo", values="value")
    .fillna(0)
)

p1_tabla["Sin emisiones"] = p1_tabla["Renovable"] + p1_tabla["Nuclear"]
p1_tabla["% renovable"] = p1_tabla["Renovable"] / p1_tabla["Sin emisiones"] * 100
p1_tabla["% nuclear"] = p1_tabla["Nuclear"] / p1_tabla["Sin emisiones"] * 100

display(p1_tabla)

In [ ]:
# Guardamos P1 para Influx/Grafana.

p1_influx = p1_tabla.reset_index().melt(
    id_vars="year",
    value_vars=["Renovable", "Nuclear", "Sin emisiones", "% renovable", "% nuclear"],
    var_name="metrica",
    value_name="valor"
)

p1_influx["fecha"] = pd.to_datetime(p1_influx["year"].astype(str) + "-01-01")
p1_influx["tecnologia"] = ""
p1_influx["territorio"] = ""
p1_influx["pais"] = ""

guardar_salida(p1_influx, "P1", "P1_renovable_nuclear")

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(p1_tabla.index, p1_tabla["Renovable"], marker="o", label="Renovable")
plt.plot(p1_tabla.index, p1_tabla["Nuclear"], marker="o", label="Nuclear")
plt.title("P1 — Renovable y nuclear")
plt.xlabel("Año")
plt.ylabel("MWh")
plt.grid(True)
plt.legend()
plt.show()

# 4. P2 — Renovable frente a generación con emisiones

### Pregunta

**Cuando aumenta la participación renovable, ¿se observa al mismo tiempo una reducción de la generación con emisiones o ambas pueden crecer en determinados periodos?**

Aquí no buscamos demostrar causalidad. Solo comparamos cómo evolucionan ambas magnitudes.

In [ ]:
p2_base = df[
    (df["geography_type"] == "system") &
    (df["endpoint"].isin([
        "evolucion_generacion_emisiones",
        "evolucion_renovable_no_renovable"
    ])) &
    (df["value_unit"] == "MWh") &
    (df["indicator_name"].isin([
        "Renovable",
        "Con emisiones de CO2 eq.",
        "Sin emisiones de CO2 eq.",
        "No renovable"
    ]))
].copy()

p2_anual = (
    p2_base.groupby(["year", "indicator_name"], as_index=False)["value"]
    .sum()
)

display(p2_anual)

In [ ]:
p2_pivot = (
    p2_anual.pivot(index="year", columns="indicator_name", values="value")
    .fillna(0)
)

if "Renovable" in p2_pivot.columns and "Con emisiones de CO2 eq." in p2_pivot.columns:
    p2_pivot["variacion_renovable_%"] = p2_pivot["Renovable"].pct_change() * 100
    p2_pivot["variacion_emisiones_%"] = p2_pivot["Con emisiones de CO2 eq."].pct_change() * 100

display(p2_pivot)

In [ ]:
p2_influx = p2_anual.copy()
p2_influx["fecha"] = pd.to_datetime(p2_influx["year"].astype(str) + "-01-01")
p2_influx["metrica"] = p2_influx["indicator_name"]
p2_influx["valor"] = p2_influx["value"]
p2_influx["tecnologia"] = ""
p2_influx["territorio"] = ""
p2_influx["pais"] = ""

guardar_salida(
    p2_influx[["fecha","metrica","valor","tecnologia","territorio","pais"]],
    "P2",
    "P2_renovable_emisiones"
)

In [ ]:
plt.figure(figsize=(10,5))

for columna in ["Renovable", "Con emisiones de CO2 eq."]:
    if columna in p2_pivot.columns:
        plt.plot(p2_pivot.index, p2_pivot[columna], marker="o", label=columna)

plt.title("P2 — Renovable frente a generación con emisiones")
plt.xlabel("Año")
plt.ylabel("MWh")
plt.grid(True)
plt.legend()
plt.show()

# 5. P3 — Tecnologías con crecimiento estructural

### Pregunta

**¿Qué tecnologías muestran un crecimiento estructural y cuáles presentan una evolución más irregular entre 2019 y 2025?**

Calcularemos:

- generación anual,
- crecimiento acumulado desde 2019,
- variación interanual,
- desviación de la variación interanual.

In [ ]:
tecnologias_p3 = [
    "Eólica", "Solar fotovoltaica", "Hidráulica",
    "Ciclo combinado", "Carbón", "Cogeneración"
]

p3 = df[
    (df["geography_type"] == "system") &
    (df["endpoint"] == "estructura_generacion") &
    (df["value_unit"] == "MWh") &
    (df["indicator_name"].isin(tecnologias_p3))
].copy()

p3_anual = (
    p3.groupby(["year", "indicator_name"], as_index=False)["value"]
    .sum()
)

p3_anual["variacion_interanual_%"] = (
    p3_anual.groupby("indicator_name")["value"].pct_change() * 100
)

p3_resumen = (
    p3_anual.groupby("indicator_name")
    .agg(
        generacion_2019=("value", "first"),
        generacion_2025=("value", "last"),
        variabilidad_variacion=("variacion_interanual_%", "std")
    )
)

p3_resumen["crecimiento_acumulado_%"] = (
    (p3_resumen["generacion_2025"] / p3_resumen["generacion_2019"]) - 1
) * 100

display(p3_resumen.sort_values("crecimiento_acumulado_%", ascending=False))

In [ ]:
p3_influx = p3_anual.copy()
p3_influx["fecha"] = pd.to_datetime(p3_influx["year"].astype(str) + "-01-01")
p3_influx["metrica"] = "Generación"
p3_influx["valor"] = p3_influx["value"]
p3_influx["tecnologia"] = p3_influx["indicator_name"]
p3_influx["territorio"] = ""
p3_influx["pais"] = ""

guardar_salida(
    p3_influx[["fecha","metrica","valor","tecnologia","territorio","pais"]],
    "P3",
    "P3_tecnologias"
)

In [ ]:
for tecnologia in tecnologias_p3:
    datos = p3_anual[p3_anual["indicator_name"] == tecnologia]
    plt.figure(figsize=(9,4))
    plt.plot(datos["year"], datos["value"], marker="o")
    plt.title(f"P3 — {tecnologia}")
    plt.xlabel("Año")
    plt.ylabel("MWh")
    plt.grid(True)
    plt.show()

# 6. P4 — Potencia instalada frente a generación

### Pregunta

**La potencia eólica y solar ha aumentado: ¿el crecimiento de la generación ha sido proporcional al aumento de la potencia instalada o cambia el rendimiento aparente de la capacidad añadida?**

Calculamos:

**generación anual / potencia instalada**

Este indicador se interpreta como **MWh por MW instalado**. No lo llamamos eficiencia técnica.

In [ ]:
tecnologias_p4 = ["Eólica", "Solar fotovoltaica"]

generacion_p4 = df[
    (df["geography_type"] == "system") &
    (df["endpoint"] == "estructura_generacion") &
    (df["value_unit"] == "MWh") &
    (df["indicator_name"].isin(tecnologias_p4))
].groupby(["year", "indicator_name"])["value"].sum().reset_index()

potencia_p4 = df[
    (df["geography_type"] == "system") &
    (df["endpoint"] == "potencia_instalada") &
    (df["value_unit"] == "MW") &
    (df["indicator_name"].isin(tecnologias_p4))
].groupby(["year", "indicator_name"])["value"].mean().reset_index()

p4 = generacion_p4.merge(
    potencia_p4,
    on=["year", "indicator_name"],
    suffixes=("_generacion", "_potencia")
)

p4["mwh_por_mw"] = p4["value_generacion"] / p4["value_potencia"]

display(p4)

In [ ]:
p4_influx = p4.copy()
p4_influx["fecha"] = pd.to_datetime(p4_influx["year"].astype(str) + "-01-01")

p4_generacion = p4_influx.assign(
    metrica="Generación",
    valor=p4_influx["value_generacion"],
    tecnologia=p4_influx["indicator_name"]
)[["fecha","metrica","valor","tecnologia"]]

p4_potencia = p4_influx.assign(
    metrica="Potencia instalada",
    valor=p4_influx["value_potencia"],
    tecnologia=p4_influx["indicator_name"]
)[["fecha","metrica","valor","tecnologia"]]

p4_ratio = p4_influx.assign(
    metrica="MWh por MW instalado",
    valor=p4_influx["mwh_por_mw"],
    tecnologia=p4_influx["indicator_name"]
)[["fecha","metrica","valor","tecnologia"]]

p4_influx_final = pd.concat([p4_generacion, p4_potencia, p4_ratio], ignore_index=True)
p4_influx_final["territorio"] = ""
p4_influx_final["pais"] = ""

guardar_salida(
    p4_influx_final,
    "P4",
    "P4_potencia_generacion"
)

In [ ]:
for tecnologia in tecnologias_p4:
    datos = p4[p4["indicator_name"] == tecnologia]
    plt.figure(figsize=(9,4))
    plt.plot(datos["year"], datos["mwh_por_mw"], marker="o")
    plt.title(f"P4 — Generación por MW instalado: {tecnologia}")
    plt.xlabel("Año")
    plt.ylabel("MWh por MW instalado")
    plt.grid(True)
    plt.show()

# 7. P5 — Generación frente a demanda por comunidad

### Pregunta

**¿Qué comunidades presentan un mayor desequilibrio entre la electricidad que generan y la que demandan, y cómo ha evolucionado ese desequilibrio desde 2019?**

Calculamos:

**saldo = generación total - demanda**

También calculamos:

**ratio = generación total / demanda**

### Control territorial

El análisis debe excluir inicialmente los territorios señalados en el documento del proyecto como problemáticos:

- Navarra
- Comunidad de Madrid
- Región de Murcia

hasta validar esos registros con la fuente de REE.

In [ ]:
territorios_excluir = [
    "Navarra",
    "Comunidad de Madrid",
    "Región de Murcia"
]

generacion_ccaa = df[
    (df["geography_type"] == "ccaa") &
    (df["endpoint"] == "estructura_generacion") &
    (df["indicator_name"] == "Generación total") &
    (df["value_unit"] == "MWh") &
    (~df["geography_name"].isin(territorios_excluir))
].groupby(["year", "geography_name"])["value"].sum().reset_index()

demanda_ccaa = df[
    (df["geography_type"] == "ccaa") &
    (df["endpoint"] == "demanda_evolucion") &
    (df["indicator_name"] == "Demanda") &
    (df["value_unit"] == "MWh") &
    (~df["geography_name"].isin(territorios_excluir))
].groupby(["year", "geography_name"])["value"].sum().reset_index()

p5 = generacion_ccaa.merge(
    demanda_ccaa,
    on=["year", "geography_name"],
    suffixes=("_generacion", "_demanda")
)

p5["saldo_generacion_demanda"] = (
    p5["value_generacion"] - p5["value_demanda"]
)

p5["ratio_generacion_demanda"] = (
    p5["value_generacion"] / p5["value_demanda"]
)

display(p5.head(30))

In [ ]:
p5_influx = pd.concat([
    p5.assign(
        metrica="Generación",
        valor=p5["value_generacion"]
    )[["year","geography_name","metrica","valor"]],
    p5.assign(
        metrica="Demanda",
        valor=p5["value_demanda"]
    )[["year","geography_name","metrica","valor"]],
    p5.assign(
        metrica="Saldo generación-demanda",
        valor=p5["saldo_generacion_demanda"]
    )[["year","geography_name","metrica","valor"]],
    p5.assign(
        metrica="Ratio generación-demanda",
        valor=p5["ratio_generacion_demanda"]
    )[["year","geography_name","metrica","valor"]]
], ignore_index=True)

p5_influx["fecha"] = pd.to_datetime(p5_influx["year"].astype(str) + "-01-01")
p5_influx["tecnologia"] = ""
p5_influx["territorio"] = p5_influx["geography_name"]
p5_influx["pais"] = ""

guardar_salida(
    p5_influx[["fecha","metrica","valor","tecnologia","territorio","pais"]],
    "P5",
    "P5_territorios"
)

In [ ]:
# Ranking de territorios según el saldo de 2025

ranking_2025 = (
    p5[p5["year"] == 2025]
    .sort_values("saldo_generacion_demanda", ascending=False)
)

display(
    ranking_2025[
        ["geography_name", "saldo_generacion_demanda", "ratio_generacion_demanda"]
    ]
)

# 8. P6 — Hidráulica frente a generación con emisiones

### Pregunta

**En los periodos en los que la generación hidráulica disminuye de forma significativa, ¿qué cambios se observan en la generación de tecnologías con emisiones?**

Aquí utilizamos datos mensuales para detectar mejor los cambios.

No afirmamos causalidad: buscamos una asociación observable.

In [ ]:
p6_base = df[
    (df["geography_type"] == "system") &
    (df["endpoint"].isin([
        "estructura_generacion",
        "evolucion_generacion_emisiones"
    ])) &
    (df["value_unit"] == "MWh") &
    (df["indicator_name"].isin([
        "Hidráulica",
        "Con emisiones de CO2 eq."
    ]))
].copy()

# Agrupamos por año y mes para trabajar a nivel mensual.

p6_base["periodo"] = p6_base["date"].dt.to_period("M").dt.to_timestamp()

p6_mensual = (
    p6_base.groupby(["periodo", "indicator_name"], as_index=False)["value"]
    .sum()
)

p6_pivot = (
    p6_mensual.pivot(index="periodo", columns="indicator_name", values="value")
    .fillna(0)
)

p6_pivot["variacion_hidraulica_%"] = p6_pivot["Hidráulica"].pct_change() * 100
p6_pivot["variacion_emisiones_%"] = p6_pivot["Con emisiones de CO2 eq."].pct_change() * 100

display(p6_pivot.head(20))

In [ ]:
p6_influx = p6_mensual.copy()
p6_influx["fecha"] = p6_influx["periodo"]
p6_influx["metrica"] = p6_influx["indicator_name"]
p6_influx["valor"] = p6_influx["value"]
p6_influx["tecnologia"] = ""
p6_influx["territorio"] = ""
p6_influx["pais"] = ""

guardar_salida(
    p6_influx[["fecha","metrica","valor","tecnologia","territorio","pais"]],
    "P6",
    "P6_hidraulica_emisiones"
)

In [ ]:
plt.figure(figsize=(11,5))
plt.plot(p6_pivot.index, p6_pivot["Hidráulica"], label="Hidráulica")
plt.plot(p6_pivot.index, p6_pivot["Con emisiones de CO2 eq."], label="Con emisiones")
plt.title("P6 — Hidráulica y generación con emisiones")
plt.xlabel("Mes")
plt.ylabel("MWh")
plt.grid(True)
plt.legend()
plt.show()

# 9. P7 — Intercambios internacionales

### Pregunta

**¿Cómo ha evolucionado el saldo de los intercambios eléctricos con Francia, Portugal, Marruecos y Andorra y en qué momentos cambia España de una posición importadora a exportadora?**

Usaremos los endpoints físicos diarios:

- `intercambios_francia_fisicos`
- `intercambios_portugal_fisicos`
- `intercambios_marruecos_fisicos`
- `intercambios_andorra_fisicos`

El indicador `saldo` será nuestra magnitud principal.

In [ ]:
paises = {
    "intercambios_francia_fisicos": "Francia",
    "intercambios_portugal_fisicos": "Portugal",
    "intercambios_marruecos_fisicos": "Marruecos",
    "intercambios_andorra_fisicos": "Andorra"
}

p7 = df[
    (df["geography_type"] == "system") &
    (df["endpoint"].isin(paises.keys())) &
    (df["indicator_name"] == "saldo") &
    (df["value_unit"] == "MWh")
].copy()

p7["pais"] = p7["endpoint"].map(paises)

p7_mensual = (
    p7.assign(periodo=p7["date"].dt.to_period("M").dt.to_timestamp())
    .groupby(["periodo", "pais"], as_index=False)["value"]
    .sum()
)

display(p7_mensual.head(30))

In [ ]:
# Detectamos cambios de signo del saldo.
# Convención usada por el dataset: se conserva el signo tal como viene de REE.

p7_mensual["signo"] = np.where(
    p7_mensual["value"] > 0, "Positivo",
    np.where(p7_mensual["value"] < 0, "Negativo", "Cero")
)

p7_mensual["cambio_signo"] = (
    p7_mensual.groupby("pais")["signo"]
    .transform(lambda x: x.ne(x.shift()).fillna(False))
)

cambios_p7 = p7_mensual[p7_mensual["cambio_signo"]].copy()

display(cambios_p7)

In [ ]:
p7_influx = p7_mensual.copy()
p7_influx["fecha"] = p7_influx["periodo"]
p7_influx["metrica"] = "Saldo"
p7_influx["valor"] = p7_influx["value"]
p7_influx["tecnologia"] = ""
p7_influx["territorio"] = ""

guardar_salida(
    p7_influx[["fecha","metrica","valor","tecnologia","territorio","pais"]],
    "P7",
    "P7_intercambios"
)

In [ ]:
for pais in p7_mensual["pais"].unique():
    datos = p7_mensual[p7_mensual["pais"] == pais]
    plt.figure(figsize=(11,4))
    plt.plot(datos["periodo"], datos["value"])
    plt.axhline(0, linewidth=1)
    plt.title(f"P7 — Saldo de intercambio con {pais}")
    plt.xlabel("Mes")
    plt.ylabel("MWh")
    plt.grid(True)
    plt.show()

# 10. Unificamos todas las salidas para InfluxDB

Ahora tenemos los resultados de P1–P7 en tablas separadas.

Las unificamos en una única tabla con esta estructura:

| columna | función |
|---|---|
| `fecha` | timestamp de InfluxDB |
| `pregunta` | P1...P7 |
| `metrica` | indicador |
| `tecnologia` | tecnología, si aplica |
| `territorio` | CCAA, si aplica |
| `pais` | país, si aplica |
| `valor` | valor numérico |

Esto es lo que vamos a enviar a InfluxDB.

In [ ]:
influx_df = pd.concat(salidas, ignore_index=True)

# Aseguramos tipos correctos
influx_df["fecha"] = pd.to_datetime(influx_df["fecha"], errors="coerce")
influx_df["valor"] = pd.to_numeric(influx_df["valor"], errors="coerce")

for columna in ["pregunta", "metrica", "tecnologia", "territorio", "pais"]:
    influx_df[columna] = influx_df[columna].fillna("").astype(str)

influx_df = influx_df.dropna(subset=["fecha", "valor"]).copy()

# Guardamos una copia local que también sirve para revisar los datos antes de enviarlos.
ruta_maestra = os.path.join(SALIDA, "REE_resultados_para_InfluxDB.csv")
influx_df.to_csv(ruta_maestra, index=False, encoding="utf-8-sig")

print("Filas preparadas para InfluxDB:", len(influx_df))
print("Archivo:", ruta_maestra)

display(influx_df.head(30))

# 11. Configuración de InfluxDB 2.x

## Opción recomendada: variables de entorno

No escribas el token directamente en el notebook si el proyecto va a compartirse.

Define estas variables antes de ejecutar la conexión:

```text
INFLUX_URL
INFLUX_TOKEN
INFLUX_ORG
INFLUX_BUCKET
```

Por ejemplo, en Linux:

```bash
export INFLUX_URL="http://localhost:8086"
export INFLUX_TOKEN="TU_TOKEN"
export INFLUX_ORG="TU_ORG"
export INFLUX_BUCKET="ree"
```

Si utilizas Windows, puedes configurar las mismas variables desde el sistema.

> El notebook no escribe nada en InfluxDB hasta que ejecutes explícitamente la celda de escritura.

In [ ]:
# Lee la configuración desde variables de entorno.

INFLUX_URL = os.getenv("INFLUX_URL", "http://localhost:8086")
INFLUX_TOKEN = os.getenv("INFLUX_TOKEN", "")
INFLUX_ORG = os.getenv("INFLUX_ORG", "")
INFLUX_BUCKET = os.getenv("INFLUX_BUCKET", "")

print("URL:", INFLUX_URL)
print("ORG configurada:", bool(INFLUX_ORG))
print("BUCKET configurado:", bool(INFLUX_BUCKET))
print("TOKEN configurado:", bool(INFLUX_TOKEN))

# 12. Escribir los resultados en InfluxDB

La estructura que se crea es:

**measurement:** `ree_analisis`

**tags:**
- `pregunta`
- `metrica`
- `tecnologia`
- `territorio`
- `pais`

**field:**
- `valor`

Ejemplo conceptual:

```text
ree_analisis,pregunta=P1,metrica=Renovable valor=123456.7 2025-01-01T00:00:00Z
```

Los tags permiten que Grafana filtre fácilmente por pregunta, tecnología, territorio o país.

In [ ]:
# Conexión y escritura.
# EJECUTA ESTA CELDA SOLO CUANDO HAYAS CONFIGURADO URL, TOKEN, ORG Y BUCKET.

from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS

if not all([INFLUX_URL, INFLUX_TOKEN, INFLUX_ORG, INFLUX_BUCKET]):
    print("Faltan datos de configuración. No se ha escrito nada en InfluxDB.")
else:
    with InfluxDBClient(
        url=INFLUX_URL,
        token=INFLUX_TOKEN,
        org=INFLUX_ORG
    ) as client:

        write_api = client.write_api(write_options=SYNCHRONOUS)

        puntos = []

        for _, fila in influx_df.iterrows():
            punto = (
                Point("ree_analisis")
                .tag("pregunta", fila["pregunta"])
                .tag("metrica", fila["metrica"])
                .tag("tecnologia", fila["tecnologia"])
                .tag("territorio", fila["territorio"])
                .tag("pais", fila["pais"])
                .field("valor", float(fila["valor"]))
                .time(fila["fecha"], WritePrecision.s)
            )
            puntos.append(punto)

        write_api.write(
            bucket=INFLUX_BUCKET,
            org=INFLUX_ORG,
            record=puntos
        )

        print("Datos enviados correctamente a InfluxDB.")
        print("Puntos enviados:", len(puntos))

# 13. Comprobar los datos desde Python

Esta celda permite comprobar que InfluxDB responde antes de entrar en Grafana.

In [ ]:
# Consulta de prueba desde Python.
# Requiere que la conexión anterior esté configurada.

from influxdb_client import InfluxDBClient

flux_prueba = f'''
from(bucket: "{INFLUX_BUCKET}")
  |> range(start: 2019-01-01, stop: 2026-01-01)
  |> filter(fn: (r) => r["_measurement"] == "ree_analisis")
  |> filter(fn: (r) => r["pregunta"] == "P1")
'''

if all([INFLUX_URL, INFLUX_TOKEN, INFLUX_ORG, INFLUX_BUCKET]):
    with InfluxDBClient(
        url=INFLUX_URL,
        token=INFLUX_TOKEN,
        org=INFLUX_ORG
    ) as client:
        resultado = client.query_api().query_data_frame(
            query=flux_prueba,
            org=INFLUX_ORG
        )
        display(resultado.head(20))
else:
    print("Configura InfluxDB antes de ejecutar esta consulta.")

# 14. Consultas Flux listas para Grafana

En Grafana:

**Add data source → InfluxDB → Flux**

Después puedes crear un panel y pegar una de las siguientes consultas.

## Panel P1 — Renovable vs nuclear

In [ ]:
from(bucket: "ree")
  |> range(start: 2019-01-01, stop: 2026-01-01)
  |> filter(fn: (r) => r["_measurement"] == "ree_analisis")
  |> filter(fn: (r) => r["pregunta"] == "P1")
  |> filter(fn: (r) => r["metrica"] == "Renovable" or r["metrica"] == "Nuclear")
  |> keep(columns: ["_time", "_value", "metrica"])
  |> pivot(rowKey:["_time"], columnKey:["metrica"], valueColumn:"_value")

## Panel P2 — Renovable vs emisiones

In [ ]:
from(bucket: "ree")
  |> range(start: 2019-01-01, stop: 2026-01-01)
  |> filter(fn: (r) => r["_measurement"] == "ree_analisis")
  |> filter(fn: (r) => r["pregunta"] == "P2")
  |> filter(fn: (r) => r["metrica"] == "Renovable" or r["metrica"] == "Con emisiones de CO2 eq.")
  |> keep(columns: ["_time", "_value", "metrica"])

## Panel P3 — Evolución de tecnologías

In [ ]:
from(bucket: "ree")
  |> range(start: 2019-01-01, stop: 2026-01-01)
  |> filter(fn: (r) => r["_measurement"] == "ree_analisis")
  |> filter(fn: (r) => r["pregunta"] == "P3")
  |> filter(fn: (r) => r["metrica"] == "Generación")
  |> keep(columns: ["_time", "_value", "tecnologia"])

## Panel P4 — Generación por MW instalado

In [ ]:
from(bucket: "ree")
  |> range(start: 2019-01-01, stop: 2026-01-01)
  |> filter(fn: (r) => r["_measurement"] == "ree_analisis")
  |> filter(fn: (r) => r["pregunta"] == "P4")
  |> filter(fn: (r) => r["metrica"] == "MWh por MW instalado")
  |> keep(columns: ["_time", "_value", "tecnologia"])

## Panel P5 — Saldo generación-demanda por territorio

En Grafana puedes añadir una variable `$territorio` para poder cambiar de comunidad.

In [ ]:
from(bucket: "ree")
  |> range(start: 2019-01-01, stop: 2026-01-01)
  |> filter(fn: (r) => r["_measurement"] == "ree_analisis")
  |> filter(fn: (r) => r["pregunta"] == "P5")
  |> filter(fn: (r) => r["metrica"] == "Saldo generación-demanda")
  |> filter(fn: (r) => r["territorio"] == "${territorio}")
  |> keep(columns: ["_time", "_value", "territorio"])

## Panel P6 — Hidráulica vs emisiones

In [ ]:
from(bucket: "ree")
  |> range(start: 2019-01-01, stop: 2026-01-01)
  |> filter(fn: (r) => r["_measurement"] == "ree_analisis")
  |> filter(fn: (r) => r["pregunta"] == "P6")
  |> filter(fn: (r) => r["metrica"] == "Hidráulica" or r["metrica"] == "Con emisiones de CO2 eq.")
  |> keep(columns: ["_time", "_value", "metrica"])

## Panel P7 — Intercambios internacionales

Puedes crear una variable `$pais` en Grafana con:

- Francia
- Portugal
- Marruecos
- Andorra

In [ ]:
from(bucket: "ree")
  |> range(start: 2019-01-01, stop: 2026-01-01)
  |> filter(fn: (r) => r["_measurement"] == "ree_analisis")
  |> filter(fn: (r) => r["pregunta"] == "P7")
  |> filter(fn: (r) => r["metrica"] == "Saldo")
  |> filter(fn: (r) => r["pais"] == "${pais}")
  |> keep(columns: ["_time", "_value", "pais"])

# 15. Crear variables en Grafana

Para que el dashboard sea interactivo, recomendamos crear estas variables:

### Variable `territorio`

Consulta:

```flux
from(bucket: "ree")
  |> range(start: 2019-01-01, stop: 2026-01-01)
  |> filter(fn: (r) => r["_measurement"] == "ree_analisis")
  |> filter(fn: (r) => r["pregunta"] == "P5")
  |> keep(columns: ["territorio"])
  |> distinct(column: "territorio")
  |> filter(fn: (r) => r["territorio"] != "")
```

### Variable `pais`

Consulta:

```flux
from(bucket: "ree")
  |> range(start: 2019-01-01, stop: 2026-01-01)
  |> filter(fn: (r) => r["_measurement"] == "ree_analisis")
  |> filter(fn: (r) => r["pregunta"] == "P7")
  |> keep(columns: ["pais"])
  |> distinct(column: "pais")
  |> filter(fn: (r) => r["pais"] != "")
```

Así podremos seleccionar una comunidad o un país desde el propio dashboard.

# 16. Dashboard recomendado en Grafana

La estructura recomendada sería:

### Fila 1 — Visión general
- P1: evolución renovable/nuclear.
- P2: renovable vs emisiones.

### Fila 2 — Tecnologías
- P3: evolución de tecnologías.
- P4: MWh por MW instalado.

### Fila 3 — Territorio
- P5: saldo generación-demanda.
- Ranking de comunidades.

### Fila 4 — Sistema eléctrico
- P6: hidráulica vs emisiones.
- P7: intercambios internacionales.

### Filtros
- Año / rango temporal.
- Comunidad autónoma.
- País.
- Tecnología.

El objetivo es que Grafana sea la parte visual final y que Jupyter sea la parte de preparación y análisis.

# 17. Comprobación final

Antes de entregar el proyecto comprobaremos:

- [ ] P0 ejecutado.
- [ ] P1–P7 calculadas.
- [ ] Tablas revisadas.
- [ ] Gráficos revisados.
- [ ] `REE_resultados_para_InfluxDB.csv` generado.
- [ ] InfluxDB recibe los puntos.
- [ ] Grafana conecta con el bucket.
- [ ] Los paneles muestran datos.
- [ ] No se mezclan MWh y MW.
- [ ] No se presentan correlaciones como causalidad.
- [ ] Los territorios problemáticos no se utilizan sin validación.